In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("books_data.csv")
print(df.head())
print("\nDataset Shape:")
print(df.shape)
print("\nDataset Information:")
print(df.info())
#finding missing values
print("\nMissing Values:")
print(df.isnull().sum())
#finding duplicate rows
print("\nDuplicate Rows:")
print(df.duplicated().sum())

                                   Title    Price Rating Availability  \
0                   A Light in the Attic  Â£51.77  Three     In stock   
1                     Tipping the Velvet  Â£53.74    One     In stock   
2                             Soumission  Â£50.10    One     In stock   
3                          Sharp Objects  Â£47.82   Four     In stock   
4  Sapiens: A Brief History of Humankind  Â£54.23   Five     In stock   

                                                 URL  
0     catalogue/a-light-in-the-attic_1000/index.html  
1        catalogue/tipping-the-velvet_999/index.html  
2                catalogue/soumission_998/index.html  
3             catalogue/sharp-objects_997/index.html  
4  catalogue/sapiens-a-brief-history-of-humankind...  

Dataset Shape:
(1000, 5)

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Ti

In [3]:
# Convert Price to numeric
df["Price"] = (
    df["Price"]
    .str.replace("Â", "", regex=False)
    .str.replace("£", "", regex=False)
    .str.strip()
    .astype(float)
)

print("\nPrice Data Type:")
print(df["Price"].dtype)


Price Data Type:
float64


In [4]:
# Price statistics
print("\nPrice Statistics:")
print(df["Price"].describe())



Price Statistics:
count    1000.00000
mean       35.07035
std        14.44669
min        10.00000
25%        22.10750
50%        35.98000
75%        47.45750
max        59.99000
Name: Price, dtype: float64


In [5]:
# Rating distribution
print("\nRating Distribution:")
print(df["Rating"].value_counts())



Rating Distribution:
Rating
One      226
Three    203
Five     196
Two      196
Four     179
Name: count, dtype: int64


In [6]:
#finding most cheapest book and expensive book
cheapest_book = df.loc[df["Price"].idxmin()]
most_expensive_book = df.loc[df["Price"].idxmax()]

print("\nCheapest Book:")
print(cheapest_book[["Title", "Price", "Rating"]])

print("\nMost Expensive Book:")
print(most_expensive_book[["Title", "Price", "Rating"]])


Cheapest Book:
Title     An Abundance of Katherines
Price                           10.0
Rating                          Five
Name: 638, dtype: object

Most Expensive Book:
Title     The Perfect Play (Play by Play #1)
Price                                  59.99
Rating                                 Three
Name: 648, dtype: object


In [7]:
#average price for higher rating
print("\nAverage Price by Rating:")
print(df.groupby("Rating")["Price"].mean().sort_values(ascending=False))
rating_counts = df["Rating"].value_counts()



Average Price by Rating:
Rating
Four     36.093296
Five     35.374490
Two      34.810918
Three    34.692020
One      34.561195
Name: Price, dtype: float64


In [8]:
# Rating counts
print("\nNumber of Books by Rating:")
print(rating_counts)


Number of Books by Rating:
Rating
One      226
Three    203
Five     196
Two      196
Four     179
Name: count, dtype: int64


# Exploratory Data Analysis — Books Dataset

## Objectives

This analysis aims to answer the following questions:

1. What is the overall price distribution of the books?
2. Which books are the cheapest and most expensive?
3. How are books distributed across different ratings?
4. Does book rating have any relationship with price?
5. Are there any unusually expensive or unusually cheap books?
6. Are there missing, duplicate, or inconsistent records?
7. What patterns can be identified from the scraped dataset?

In [9]:
Q1 = df["Price"].quantile(0.25)
Q3 = df["Price"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = df[
    (df["Price"] < lower_limit) |
    (df["Price"] > upper_limit)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Limit:", lower_limit)
print("Upper Limit:", upper_limit)

print("\nNumber of Price Outliers:", len(outliers))

Q1: 22.1075
Q3: 47.457499999999996
IQR: 25.349999999999994
Lower Limit: -15.91749999999999
Upper Limit: 85.48249999999999

Number of Price Outliers: 0


In [10]:
outliers[["Title", "Price", "Rating"]].head(10)

,Title,Price,Rating


In [17]:
print("Number of price outliers:", len(outliers))

Number of price outliers: 0


In [18]:
rating_order = ["One", "Two", "Three", "Four", "Five"]

rating_analysis = (
    df.groupby("Rating")["Price"]
    .agg(["count", "mean", "median", "min", "max"])
    .reindex(rating_order)
)

rating_analysis

,count,mean,median,min,max
Rating,,,,,
One,226,34.561195,34.770,10.40,59.64
Two,196,34.810918,36.215,10.02,59.95
Three,203,34.692020,33.780,10.16,59.99
Four,179,36.093296,37.800,10.01,59.45
Five,196,35.374490,36.900,10.00,59.92


In [11]:
#rating vs price
rating_order = ["One", "Two", "Three", "Four", "Five"]

rating_analysis = (
    df.groupby("Rating")["Price"]
    .agg(["count", "mean", "median", "min", "max"])
    .reindex(rating_order)
)

rating_analysis

,count,mean,median,min,max
Rating,,,,,
One,226,34.561195,34.770,10.40,59.64
Two,196,34.810918,36.215,10.02,59.95
Three,203,34.692020,33.780,10.16,59.99
Four,179,36.093296,37.800,10.01,59.45
Five,196,35.374490,36.900,10.00,59.92


### Rating and Price Analysis

The table above compares the number of books, average price, median price,
minimum price, and maximum price across different rating categories.

This analysis helps determine whether book ratings are associated with
differences in pricing.

In [12]:
#converting rating to numbers
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["Rating_Numeric"] = df["Rating"].map(rating_map)

df[["Rating", "Rating_Numeric"]].head()

,Rating,Rating_Numeric
0,Three,3
1,One,1
2,One,1
3,Four,4
4,Five,5


In [13]:
#testing whether proce and rating are related
correlation = df["Price"].corr(df["Rating_Numeric"])

print("Correlation between Price and Rating:", correlation)

Correlation between Price and Rating: 0.028166239485872935


### price and rating correlation
with this correlation it conclludes clearly that there is very little linear relationship between book price and rating in this dataset.so there is no necessary of expensive books having higher rating and viceversa

In [14]:
#checking data consistency
#availability
print("Availability values:")
print(df["Availability"].value_counts())

Availability values:
Availability
In stock    1000
Name: count, dtype: int64


In [15]:
#rating
print("\nRating values:")
print(df["Rating"].value_counts())



Rating values:
Rating
One      226
Three    203
Five     196
Two      196
Four     179
Name: count, dtype: int64


In [16]:
#price
print("\nPrice range:")
print("Minimum:", df["Price"].min())
print("Maximum:", df["Price"].max())


Price range:
Minimum: 10.0
Maximum: 59.99


EDA NOTEBOOK
│
├── 1. Project Objective
│
├── 2. Analytical Questions
│
├── 3. Import Libraries
│
├── 4. Load Dataset
│
├── 5. Dataset Overview
│     ├── Shape
│     ├── Columns
│     └── Data Types
│
├── 6. Data Quality
│     ├── Missing Values
│     ├── Duplicates
│     └── Consistency
│
├── 7. Data Cleaning
│     └── Price conversion
│
├── 8. Descriptive Statistics
│
├── 9. Rating Distribution
│
├── 10. Cheapest & Most Expensive Books
│
├── 11. Price by Rating
│
├── 12. Outlier Detection
│
├── 13. Rating → Numeric Conversion
│
├── 14. Price vs Rating Correlation
│
└── 15. Key Findings

# Key Findings

1. The dataset contains 1,000 books collected through web scraping.

2. The dataset was checked for missing values and duplicate records.

3. Book prices were cleaned and converted from text into numerical values
   for statistical analysis.

4. Descriptive statistics were used to understand the distribution of book
   prices.

5. The books were analyzed across five rating categories: One, Two, Three,
   Four, and Five.

6. IQR-based analysis was performed to detect unusually priced books, and
   no price outliers were identified using this criterion.

7. The correlation between price and rating was approximately 0.028,
   indicating a very weak linear relationship.

8. Therefore, a higher book rating does not necessarily correspond to a
   higher book price in this dataset.

9. Rating and availability categories were also checked for consistency.

In [1]:
import pandas as pd

df = pd.read_csv("books_data.csv")

df["Price"] = (
    df["Price"]
    .astype(str)
    .str.replace("£", "", regex=False)
    .str.replace("Â", "", regex=False)
    .astype(float)
)

df.to_csv("books_data_cleaned.csv", index=False)

print(df.head())
print(df.dtypes)

                                   Title  Price Rating Availability  \
0                   A Light in the Attic  51.77  Three     In stock   
1                     Tipping the Velvet  53.74    One     In stock   
2                             Soumission  50.10    One     In stock   
3                          Sharp Objects  47.82   Four     In stock   
4  Sapiens: A Brief History of Humankind  54.23   Five     In stock   

                                                 URL  
0     catalogue/a-light-in-the-attic_1000/index.html  
1        catalogue/tipping-the-velvet_999/index.html  
2                catalogue/soumission_998/index.html  
3             catalogue/sharp-objects_997/index.html  
4  catalogue/sapiens-a-brief-history-of-humankind...  
Title               str
Price           float64
Rating              str
Availability        str
URL                 str
dtype: object
